# TP2 — Prise en main de Simu5G : analyse des résultats

Ce notebook lit les fichiers CSV produits par `tp-export` et trace les KPI.
Exécutez les cellules dans l'ordre (Shift+Entrée). Les zones `# TODO` sont à compléter.

In [ ]:
import sys; sys.path.append('/tp/common')
import pandas as pd, matplotlib.pyplot as plt
from tpanalyse import load_scalars, kpi_par_run
plt.rcParams['figure.figsize'] = (9, 4.5)

## 1. Un seul run : quels KPI sont disponibles ?
Après `tp-run SingleCell-DL` puis `tp-export results results.csv` :

In [ ]:
sca, iv = load_scalars('results.csv')
print(sca.name.unique())          # tous les scalaires enregistrés
sca[sca.name.str.startswith('voIP')].head(20)

**Q2.1** — Quels sont, pour chaque UE, le délai moyen (`voIPFrameDelay:mean`), la perte (`voIPFrameLoss:mean`) et le MOS (`voIPMos:mean`) ? Lequel des UE est le plus mal servi, et pourquoi (regardez sa position dans le .ini) ?

In [ ]:
kpi = sca[sca.name.isin(['voIPFrameDelay:mean','voIPFrameLoss:mean','voIPMos:mean'])]
kpi.pivot_table(index='module', columns='name', values='value')

## 2. Balayage de charge (config `Charge-DL`, runs 0..4)
Après `tp-run Charge-DL 0..4` et `tp-export results/Charge-DL charge.csv` :

In [ ]:
sca, iv = load_scalars('charge.csv')
delay = kpi_par_run(sca, iv, 'voIPFrameDelay:mean', module_filter='ue[')
loss  = kpi_par_run(sca, iv, 'voIPFrameLoss:mean',  module_filter='ue[')
res = delay.merge(loss, on=['run','numUEs']).sort_values('numUEs')
res

In [ ]:
fig, ax = plt.subplots(1, 2)
ax[0].plot(res.numUEs, res['voIPFrameDelay:mean']*1000, 'o-'); ax[0].set_xlabel('Nombre d\'UE'); ax[0].set_ylabel('Délai moyen (ms)'); ax[0].grid(alpha=.3)
ax[1].plot(res.numUEs, res['voIPFrameLoss:mean']*100, 's-r'); ax[1].set_xlabel('Nombre d\'UE'); ax[1].set_ylabel('Perte (%)'); ax[1].grid(alpha=.3)
plt.tight_layout()

**Q2.2** — À partir de quelle charge le délai décroche-t-il ? Reliez ce seuil au nombre de RB (`numBands = 25`) et au débit d'un flux VoIP (≈ 40 octets toutes les 20 ms). Combien de flux VoIP la cellule peut-elle théoriquement porter ?

**Q2.3** — Relancez le balayage avec `numBands = 50` puis `100` (modifiez le .ini, exportez dans `charge50.csv` / `charge100.csv`) et superposez les trois courbes ci-dessous.

In [ ]:
# TODO : charger charge50.csv et charge100.csv, tracer les 3 courbes délai vs numUEs sur le même graphe


## 3. Mobilité (config `Mobile-DL`, runs 0..3)

In [ ]:
sca, iv = load_scalars('mobile.csv')
mob = kpi_par_run(sca, iv, 'voIPFrameDelay:mean', module_filter='ue[').sort_values('speed')
plt.plot(mob.speed*3.6, mob['voIPFrameDelay:mean']*1000, 'o-'); plt.xlabel('Vitesse (km/h)'); plt.ylabel('Délai moyen (ms)'); plt.grid(alpha=.3)

**Q2.4** — La vitesse a-t-elle un effet sur le délai ? Sur quel mécanisme de la chaîne (CQI/AMC, HARQ) joue-t-elle ? Que faudrait-il ajouter au scénario pour observer un vrai handover ?

## 4. Synthèse (à rédiger ici, 10 lignes max)
- Capacité VoIP de la cellule pour 5 / 10 / 20 MHz :
- Critère qui limite en premier (délai ou perte) :
- Ce que vous avez appris sur le lien .ini → résultats :